# 1.0 IMPORT LIBRARIES

In [ ]:
from pyspark.sql.types import StringType, IntegerType, FloatType, BooleanType, DateType, TimestampType, StructField, StructType
from pyspark.sql.functions import col, hour, second, datediff

# 2.0 FILE PATHS IN THE DATABASE FILE SYSTEM

In [ ]:
payments_data = 'dbfs:/FileStore/project4/payments.csv'
riders_data = 'dbfs:/FileStore/project4/riders.csv'
stations_data = 'dbfs:/FileStore/project4/stations.csv'
trips_data = 'dbfs:/FileStore/project4/trips.csv'

# 3.0 DATA EXTRACTION

In [ ]:
def read_csv(csv_data, schema):
    return spark.read.option('inferSchema', 'false').option('header','false').csv(csv_data, schema)

## 3.1 'Payments' data extraction

In [ ]:
payments_table = StructType([
    StructField('payment_id', IntegerType(), nullable=False),
    StructField('date', DateType(), nullable=False),
    StructField('amount', FloatType(), nullable=False),
    StructField('rider_id', IntegerType(), nullable=False)
])
payments_df = read_csv(payments_data, payments_table)

display(payments_df)

In [ ]:
payments_df.write.format('delta').mode('overwrite').save("/delta/project4/bronze_payments")

## 3.2 'Riders' data extraction

In [ ]:
riders_table = StructType([
    StructField('rider_id', IntegerType(), nullable=False),
    StructField('first', StringType(), nullable=False),
    StructField('last', StringType(), nullable=False),
    StructField('address', StringType(), nullable=False),
    StructField('birthday', DateType(), nullable=False),
    StructField('account_start_date', DateType(), nullable=False),
    StructField('account_end_date', DateType(), nullable=True),
    StructField('is_member', BooleanType(), nullable=False)
])
riders_df = read_csv(riders_data, riders_table)

display(riders_df)

In [ ]:
riders_df.write.format('delta').mode('overwrite').save("/delta/project4/bronze_riders")

## 3.3 'Stations' data extraction

In [ ]:
stations_table = StructType([
    StructField('station_id', StringType(), nullable=False),
    StructField('name', StringType(), nullable=False),
    StructField('latitude', FloatType(), nullable=False),
    StructField('longitude', FloatType(), nullable=False)
])
stations_df = read_csv(stations_data, stations_table)

display(stations_df)

In [ ]:
stations_df.write.format('delta').mode('overwrite').save("/delta/project4/bronze_stations")

## 3.4 'Trips' data extraction

In [ ]:
trips_table = StructType([
    StructField('trip_id', StringType(), nullable=False),
    StructField('rideable_type', StringType(), nullable=False),
    StructField('start_at', TimestampType(), nullable=False),
    StructField('ended_at', TimestampType(), nullable=False),
    StructField('start_station_id', StringType(), nullable=False),
    StructField('end_station_id', StringType(), nullable=False),
    StructField('rider_id', IntegerType(), nullable=False)
])
trips_df = read_csv(trips_data, trips_table)

display(trips_df)

In [ ]:
trips_df.write.format('delta').mode('overwrite').save("/delta/project4/bronze_trips")

# 4.0 BRONZE DATA STORE: CREATING DELTA TABLES FROM DELTA FILES

In [ ]:
spark.sql(f"CREATE TABLE IF NOT EXISTS payments USING DELTA LOCATION '/delta/project4/bronze_payments'")
spark.sql(f"CREATE TABLE IF NOT EXISTS riders USING DELTA LOCATION '/delta/project4/bronze_riders'")
spark.sql(f"CREATE TABLE IF NOT EXISTS stations USING DELTA LOCATION '/delta/project4/bronze_stations'")
spark.sql(f"CREATE TABLE IF NOT EXISTS trips USING DELTA LOCATION '/delta/project4/bronze_trips'")

# 5.0 GOLD DATA STORE: CREATING FACT AND DIMENSION TABLES FROM DELTA TABLES

In [ ]:
payments_table_df = spark.table("default.payments")
riders_table_df = spark.table("default.riders")
stations_table_df = spark.table("default.stations")
trips_table_df = spark.table("default.trips")

## 5.1 Creating Date Dimension Table

In [ ]:
distinct_payment_dates = payments_table_df.select(col('date').alias('calendarDate')).distinct()
distinct_riders_birth_dates = riders_table_df.select(col('birthday').alias('calendarDate')).distinct()
distinct_account_start_dates = riders_table_df.select(col('account_start_date').alias('calendarDate')).distinct()
distinct_account_end_dates = riders_table_df.select(col('account_end_date').alias('calendarDate')).distinct()

all_distinct_dates = distinct_payment_dates.unionByName(distinct_riders_birth_dates).unionByName(distinct_account_start_dates).unionByName(distinct_account_end_dates)
all_distinct_dates.createOrReplaceTempView('all_distinct_dates')
all_distinct_dates.display()

In [ ]:
dimDate_df = spark.sql("""
    select
      calendarDate,
      year(calendarDate) AS CalendarYear,
      date_format(calendarDate, 'MMMM') as CalendarMonth,
      month(calendarDate) as MonthOfYear,
      date_format(calendarDate, 'EEEE') as CalendarDay,
      dayofweek(calendarDate) AS DayOfWeek,
      weekday(calendarDate) + 1 as DayOfWeekStartMonday,
      case
        when weekday(calendarDate) < 5 then 'Y'
        else 'N'
      end as IsWeekDay,
      dayofmonth(calendarDate) as DayOfMonth,
      case
        when calendarDate = last_day(calendarDate) then 'Y'
        else 'N'
      end as IsLastDayOfMonth,
      dayofyear(calendarDate) as DayOfYear,
      weekofyear(calendarDate) as WeekOfYearIso,
      quarter(calendarDate) as QuarterOfYear
    from
      all_distinct_dates
    order by
      calendarDate
""")
dimDate_df.write.format('delta').mode('overwrite').saveAsTable('dimDate')

## 5.2 Creating Stations Dimension Table

In [ ]:
stations_table_df.write.format('delta').mode('overwrite').saveAsTable('dimStations')

## 5.3 Creating Riders Dimension Table

In [ ]:
riders_table_df.write.format('delta').mode('overwrite').saveAsTable('dimRiders')

## 5.4 Creating Payments Fact Table

In [ ]:
payments_table_df.write.format('delta').mode('overwrite').saveAsTable('factPayments')

## 5.5 Creating Trips Fact Table

In [ ]:
trips_table_df = trips_table_df.join(riders_table_df, 'rider_id', how='inner')\
.withColumn('duration_seconds', col('ended_at').cast('long') - col('start_at').cast('long'))\
.withColumn('rider_age', (datediff(col('start_at'), col('birthday'))/365.25).cast('integer'))\
.withColumn('trip_started_at', col('start_at').cast('date'))\
.withColumn('trip_ended_at',col('ended_at').cast('date'))\
.withColumn('trip_start_hour', hour(col('start_at')))\
.withColumn('trip_end_hour', hour(col('ended_at')))\
.select('trip_id,rideable_type,duration_seconds,rider_age,trip_started_at,trip_ended_at,trip_start_hour,trip_end_hour,start_station_id,end_station_id,rider_id'.split(','))

trips_table_df.write.format('delta').mode('overwrite').saveAsTable('factTrips')